# 9.14 有機率輸出，還要怎樣訓練與校準？


假設一百次被模型報為「約八成」的判斷，只對四十次。輸出雖然是合法機率，使用者仍會誤判風險。我們要的兩件事是：選對答案，以及讓報出的機率與留出題上的成敗相稱。

先用有正解的例子訓練。Choice 可以用交叉熵 `-log(p_correct)`；yes／no 用二元交叉熵；有限等級則先把每一級寫成明確規準，再對該級做分類。例如「0＝沒有必要資訊、1＝只有任務、2＝任務及所需編號都有」是我們人工訂定的三級資訊規準。**Score** 在這裡指有次序的有限等級分布，不是任意生成一個沒有標準的滿意度小數。

若有可靠的目標分布 q，交叉熵可寫成 `-sum(q*log(p))`；只要 q 是教師猜測，優化目標就首先是貼近教師。交叉熵和 Brier score 都是 proper scoring rule：在理想的真實條件分布、期望評分及可表達的條件下，如實報該分布能使期望損失最小。有限資料、模型限制與標籤錯誤會破壞「訓練好了便可靠」的推論。

下面用同一個正解檢查 Brier 的方向。向量各位置依序代表卡片、地址、其他；one-hot 真值是第一類。此處採三類平方差**總和**，不再除類別數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

y = torch.tensor([1.0, 0.0, 0.0])
for p in [torch.tensor([0.8, 0.1, 0.1]), torch.tensor([0.2, 0.7, 0.1])]:
    print("CE", float(-p[0].log()), "Brier", float(((p - y) ** 2).sum()))

前者 Brier 是0.06，後者是1.14，這一題比較偏好前者；它不是整體校準評估。[9.10](https://birdhackor.github.io/tiny-perceptron-vlm/9.10.html)的溫度縮放可在獨立 calibration 資料上比較事先列出的正數 T，以平均交叉熵最低者為準，再把 logits 除以選定的 T 做 softmax。這會改變尖銳程度，對固定候選不改 argmax，因此不能修復選錯類別本身。

實際資料至少分開訓練、calibration、policy 與最後 test：前者更新權重，接著選溫度，再選行動門檻，最後只報凍結後成績。樣本有限可以清楚記錄重用與不確定性，不能一邊看 test 一邊調 T。重排、缺候選與新措辭也各自報分，平均校準好不保證每個子群都好。

練習：一個永遠報三類均勻的模型可能不過度自信，但能因此說它有用嗎？還要看辨識能力與任務成本；校準與正確率回答不同問題。
